# 02 - Frame Sampling
Extract candidate frames from videos for labeling.

In [ ]:
# ===== CONFIGURATION =====
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/LightningPoseTrack.git"
GIT_BRANCH = "main"

DRIVE_RAW_VIDEOS = "/content/drive/My Drive/PigBehavior/raw_videos"
DRIVE_LABELED = "/content/drive/My Drive/PigBehavior/labeled_frames"

# Sampling settings
FRAMES_PER_VIDEO = 30       # total target: ~300 frames across 10+ videos
SAMPLING_METHOD = "uniform"  # "uniform", "random", or "motion"
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install --quiet opencv-python pandas numpy

In [ ]:
import cv2
from pathlib import Path
from src.pose.frame_sampler import sample_uniform, sample_random, sample_motion_based, save_frame
from src.io.video_inventory import scan_videos, parse_camera_from_filename

df = scan_videos(DRIVE_RAW_VIDEOS)
print(f"Found {len(df)} videos")

samplers = {
    "uniform": sample_uniform,
    "random": sample_random,
    "motion": sample_motion_based,
}
sampler = samplers.get(SAMPLING_METHOD, sample_uniform)

total_saved = 0
for _, row in df.iterrows():
    video_path = Path(DRIVE_RAW_VIDEOS) / row["path"]
    camera = row["camera"]
    session = row["session"]
    stem = Path(row["filename"]).stem
    frames = sampler(str(video_path), FRAMES_PER_VIDEO)
    for frame_idx, frame in frames:
        fname = save_frame(DRIVE_LABELED, session, camera, stem, frame_idx, frame)
        total_saved += 1
    print(f"{row['filename']}: saved {len(frames)} frames")

print(f"\nTotal frames saved: {total_saved} to {DRIVE_LABELED}")

In [ ]:
print("=" * 60)
print("NEXT STEPS:")
print("1. Download the labeled_frames folder from Google Drive")
print("2. Label keypoints using Lightning Pose labeling tool or LabelMe")
print("3. Upload labeled data back to Drive under labeled_frames/")
print("4. Proceed to notebook 03_Pose_Training.ipynb")